# Setup Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Clusters_amritap1")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.memory", "10g")
    .config("spark.executor.cores", "1")
    .config("spark.executor.instances", "14")
    .config("spark.serializer","org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer","1g")
    .config("spark.driver.maxResultSize","8g")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/12 18:22:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Load Data

In [2]:
commentsDF = spark.read.parquet("/project/macs40123/amritap1/macs-40123-amritapathak1/final_data_with_id/comments")

In [3]:
submissionsDF = spark.read.parquet("/project/macs40123/amritap1/macs-40123-amritapathak1/final_data_with_id/submissions")

# EDA

## Dataset Overview

## Dataset Overview

The dataset contains approximately **65 million total records**, consisting of roughly **62.8 million comments** and **2.0 million submissions**. This substantial imbalance reflects the conversational structure of the platform, where submissions serve primarily as discussion anchors while comments capture the bulk of user interaction.

The data spans a multi-year period from **early 2020 through the end of 2022**, encompassing both routine market activity and periods of extreme volatility. This temporal coverage allows for analysis across stable and highly coordinated phases of online trading behavior.

Given the extreme scale and structural imbalance of the dataset, comments and submissions are treated as analytically distinct units, and all exploratory analysis is conducted using **distributed processing with Apache Spark** to ensure computational feasibility.


### Dataset Schemas

### Dataset Schemas

The dataset schema includes both structural and behavioral variables relevant to large-scale discourse analysis. Key fields include timestamps (`created_utc`) that enable temporal aggregation, engagement signals (`score`) that capture collective evaluation, author identifiers that support participation-level analysis, and textual fields (`body`, `title`) that allow for downstream linguistic and sentiment-based methods.

Not all schema fields are used directly in subsequent analysis; instead, emphasis is placed on variables that reflect interaction dynamics and communicative content. This selective focus ensures that later modeling steps are grounded in features that meaningfully capture collective behavior rather than platform-specific metadata.

In [19]:
print("=" * 80)
print("COMMENTS SCHEMA")
print("=" * 80)
commentsDF.printSchema()

COMMENTS SCHEMA
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- author_created_utc: long (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- body: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- score: long (nullable = true)



In [20]:
print("\n" + "=" * 80)
print("SUBMISSIONS SCHEMA")
print("=" * 80)
submissionsDF.printSchema()


SUBMISSIONS SCHEMA
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- title: string (nullable = true)



### Dataset Counts

In [21]:
comments_count = commentsDF.count()
submissions_count = submissionsDF.count()

print("=" * 80)
print("DATASET SAMPLE COUNTS")
print("=" * 80)
print(f"Total Comments:    {comments_count:,}")
print(f"Total Submissions: {submissions_count:,}")
print(f"Total Samples:     {comments_count + submissions_count:,}")
print("=" * 80)

DATASET SAMPLE COUNTS
Total Comments:    62,790,822
Total Submissions: 1,955,840
Total Samples:     64,746,662


## Comments Dataset Analysis

Because comments constitute the primary site of interaction and collective response on the platform, the remainder of the exploratory analysis focuses on their statistical and behavioral properties. Unlike submissions, which are relatively sparse and act mainly as organizational nodes, comments reflect real-time reactions, amplification, and coordination among users.

### Numeric statistics for comments

In [22]:
print("=" * 80)
print("COMMENTS: NUMERIC STATISTICS")
print("=" * 80)

comments_stats = commentsDF.select(
    F.count("*").alias("total_count"),
    F.countDistinct("author").alias("unique_authors"),
    F.min("created_utc").alias("min_created_utc"),
    F.max("created_utc").alias("max_created_utc"),
    F.avg("score").alias("avg_score"),
    F.stddev("score").alias("std_score"),
    F.min("score").alias("min_score"),
    F.max("score").alias("max_score"),
    F.avg(F.length("body")).alias("avg_body_length"),
    F.min(F.length("body")).alias("min_body_length"),
    F.max(F.length("body")).alias("max_body_length")
).collect()[0]

print(f"Total Records:           {comments_stats['total_count']:,}")
print(f"Unique Authors:          {comments_stats['unique_authors']:,}")
print(f"\nTime Range:")
print(f"  Earliest:              {comments_stats['min_created_utc']}")
print(f"  Latest:                {comments_stats['max_created_utc']}")
print(f"\nScore Statistics:")
print(f"  Average:               {comments_stats['avg_score']:.2f}")
print(f"  Std Dev:               {comments_stats['std_score']:.2f}")
print(f"  Min:                   {comments_stats['min_score']}")
print(f"  Max:                   {comments_stats['max_score']}")
print(f"\nBody Length (characters):")
print(f"  Average:               {comments_stats['avg_body_length']:.2f}")
print(f"  Min:                   {comments_stats['min_body_length']}")
print(f"  Max:                   {comments_stats['max_body_length']}")
print("=" * 80)

COMMENTS: NUMERIC STATISTICS


[Stage 49:>                                                         (0 + 8) / 9]

Total Records:           62,790,822
Unique Authors:          1,704,648

Time Range:
  Earliest:              1577836804
  Latest:                1672531193

Score Statistics:
  Average:               6.39
  Std Dev:               90.72
  Min:                   -7849
  Max:                   71870

Body Length (characters):
  Average:               75.42
  Min:                   0
  Max:                   10147


The comments dataset contains approximately **62.8 million records** authored by over **1.7 million unique users**, indicating both large-scale participation and substantial heterogeneity in activity levels. The temporal range mirrors that of the full dataset, spanning from early 2020 through the end of 2022.

Score statistics reveal a highly skewed distribution, with a relatively low average score but an extremely large standard deviation. The presence of both large negative and large positive outliers suggests that collective evaluation operates through rare but intense episodes of amplification and downvoting, rather than uniform engagement. As a result, the mean score is not representative of typical comment reception.

Comment body length further reflects the reactive nature of discourse. While the average comment is relatively short, the distribution includes both zero-length entries—likely corresponding to deleted or placeholder comments—and a long tail of unusually long posts. This pattern indicates that most interaction is brief and expressive, punctuated by occasional extended narratives or explanations.


### Score distribution percentiles for comments

In [23]:
print("\nCOMMENTS: SCORE DISTRIBUTION (Percentiles)")
print("=" * 80)

score_percentiles = commentsDF.select(
    F.expr("percentile_approx(score, 0.25)").alias("p25"),
    F.expr("percentile_approx(score, 0.50)").alias("p50_median"),
    F.expr("percentile_approx(score, 0.75)").alias("p75"),
    F.expr("percentile_approx(score, 0.90)").alias("p90"),
    F.expr("percentile_approx(score, 0.95)").alias("p95"),
    F.expr("percentile_approx(score, 0.99)").alias("p99")
).collect()[0]

print(f"25th Percentile:         {score_percentiles['p25']}")
print(f"50th Percentile (Median): {score_percentiles['p50_median']}")
print(f"75th Percentile:         {score_percentiles['p75']}")
print(f"90th Percentile:         {score_percentiles['p90']}")
print(f"95th Percentile:         {score_percentiles['p95']}")
print(f"99th Percentile:         {score_percentiles['p99']}")
print("=" * 80)


COMMENTS: SCORE DISTRIBUTION (Percentiles)


[Stage 55:>                                                         (0 + 1) / 1]

25th Percentile:         1
50th Percentile (Median): 2
75th Percentile:         3
90th Percentile:         9
95th Percentile:         15
99th Percentile:         62


An examination of score percentiles highlights the extreme concentration of attention within the comments dataset. The median score is very low, indicating that most comments receive little to no engagement, while the upper percentiles capture disproportionately large values.

This pattern reflects a highly unequal attention economy in which a small fraction of comments account for a large share of collective response. Such skewness suggests that engagement dynamics are driven by episodic bursts of visibility rather than steady accumulation, motivating the use of robust and distribution-aware methods in later analysis.


### Missing/null values analysis for comments

In [24]:
print("\nCOMMENTS: MISSING VALUES ANALYSIS")
print("=" * 80)

null_counts = commentsDF.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in commentsDF.columns
]).collect()[0]

for col_name in commentsDF.columns:
    null_count = null_counts[col_name]
    null_pct = (null_count / comments_count) * 100 if comments_count > 0 else 0
    print(f"{col_name:30s} | Nulls: {null_count:12,} ({null_pct:6.2f}%)")
print("=" * 80)


COMMENTS: MISSING VALUES ANALYSIS


[Stage 56:==================================================>     (48 + 5) / 53]

id                             | Nulls:            0 (  0.00%)
author                         | Nulls:            0 (  0.00%)
author_created_utc             | Nulls:   20,678,542 ( 32.93%)
author_flair_text              | Nulls:   49,130,549 ( 78.24%)
body                           | Nulls:            0 (  0.00%)
controversiality               | Nulls:            0 (  0.00%)
created_utc                    | Nulls:            0 (  0.00%)
is_submitter                   | Nulls:            0 (  0.00%)
score                          | Nulls:            0 (  0.00%)


### Top authors by comment count

In [25]:
print("\nCOMMENTS: TOP 20 AUTHORS BY COMMENT COUNT")
print("=" * 80)

top_authors = commentsDF.groupBy("author").agg(
    F.count("*").alias("comment_count"),
    F.avg("score").alias("avg_score")
).orderBy(F.desc("comment_count")).limit(20)

top_authors.show(20, truncate=False)


COMMENTS: TOP 20 AUTHORS BY COMMENT COUNT


[Stage 59:===================================================>    (49 + 4) / 53]

+--------------------+-------------+------------------+
|author              |comment_count|avg_score         |
+--------------------+-------------+------------------+
|[deleted]           |12135451     |2.9451880280345577|
|VisualMod           |589431       |1.7654534627462757|
|AutoModerator       |526362       |2.4516397460302986|
|cashflow_           |42867        |2.63095154781067  |
|TJMBeav             |39091        |1.6761147067099844|
|Zachjsrf            |38864        |3.4198744339234253|
|Spiritual_Extreme_81|38276        |2.916422823701536 |
|-C20H24O3           |37958        |3.328099478370831 |
|GoBeaversOSU        |34439        |4.751589767414849 |
|-Dreamville-        |31736        |3.104581547769095 |
|freehouse_throwaway |30397        |6.302398262986479 |
|EnginrA             |29373        |2.800428965376366 |
|WSBVoteBot          |26632        |1.0307524782216881|
|nosalute            |25277        |8.31851089923646  |
|KingSurfer2001      |24963        |2.3965468893

## Submissions Dataset Analysis

### Numeric statistics for submissions

In [26]:
print("=" * 80)
print("SUBMISSIONS: NUMERIC STATISTICS")
print("=" * 80)

submissions_stats = submissionsDF.select(
    F.count("*").alias("total_count"),
    F.countDistinct("author").alias("unique_authors"),
    F.min("created_utc").alias("min_created_utc"),
    F.max("created_utc").alias("max_created_utc"),
    F.avg("score").alias("avg_score"),
    F.stddev("score").alias("std_score"),
    F.min("score").alias("min_score"),
    F.max("score").alias("max_score"),
    F.avg("num_comments").alias("avg_num_comments"),
    F.stddev("num_comments").alias("std_num_comments"),
    F.min("num_comments").alias("min_num_comments"),
    F.max("num_comments").alias("max_num_comments"),
    F.avg(F.length("title")).alias("avg_title_length"),
    F.avg(F.length("selftext")).alias("avg_selftext_length")
).collect()[0]

print(f"Total Records:           {submissions_stats['total_count']:,}")
print(f"Unique Authors:          {submissions_stats['unique_authors']:,}")
print(f"\nTime Range:")
print(f"  Earliest:              {submissions_stats['min_created_utc']}")
print(f"  Latest:                {submissions_stats['max_created_utc']}")
print(f"\nScore Statistics:")
print(f"  Average:               {submissions_stats['avg_score']:.2f}")
print(f"  Std Dev:               {submissions_stats['std_score']:.2f}")
print(f"  Min:                   {submissions_stats['min_score']}")
print(f"  Max:                   {submissions_stats['max_score']}")
print(f"\nNumber of Comments Statistics:")
print(f"  Average:               {submissions_stats['avg_num_comments']:.2f}")
print(f"  Std Dev:               {submissions_stats['std_num_comments']:.2f}")
print(f"  Min:                   {submissions_stats['min_num_comments']}")
print(f"  Max:                   {submissions_stats['max_num_comments']}")
print(f"\nText Length (characters):")
print(f"  Avg Title Length:      {submissions_stats['avg_title_length']:.2f}")
print(f"  Avg Selftext Length:   {submissions_stats['avg_selftext_length']:.2f}")
print("=" * 80)

SUBMISSIONS: NUMERIC STATISTICS


[Stage 62:==================================================>       (7 + 1) / 8]

Total Records:           1,955,840
Unique Authors:          668,271

Time Range:
  Earliest:              1577836853
  Latest:                1672530717

Score Statistics:
  Average:               109.69
  Std Dev:               1984.82
  Min:                   0
  Max:                   417276

Number of Comments Statistics:
  Average:               29.74
  Std Dev:               751.01
  Min:                   0
  Max:                   100002

Text Length (characters):
  Avg Title Length:      49.89
  Avg Selftext Length:   63.30


### Score and num_comments distribution percentiles for submissions

In [27]:
print("\nSUBMISSIONS: SCORE DISTRIBUTION (Percentiles)")
print("=" * 80)

submissions_score_percentiles = submissionsDF.select(
    F.expr("percentile_approx(score, 0.25)").alias("p25"),
    F.expr("percentile_approx(score, 0.50)").alias("p50_median"),
    F.expr("percentile_approx(score, 0.75)").alias("p75"),
    F.expr("percentile_approx(score, 0.90)").alias("p90"),
    F.expr("percentile_approx(score, 0.95)").alias("p95"),
    F.expr("percentile_approx(score, 0.99)").alias("p99")
).collect()[0]

print(f"25th Percentile:         {submissions_score_percentiles['p25']}")
print(f"50th Percentile (Median): {submissions_score_percentiles['p50_median']}")
print(f"75th Percentile:         {submissions_score_percentiles['p75']}")
print(f"90th Percentile:         {submissions_score_percentiles['p90']}")
print(f"95th Percentile:         {submissions_score_percentiles['p95']}")
print(f"99th Percentile:         {submissions_score_percentiles['p99']}")


SUBMISSIONS: SCORE DISTRIBUTION (Percentiles)


[Stage 68:=============================>                            (4 + 4) / 8]

25th Percentile:         1
50th Percentile (Median): 1
75th Percentile:         2
90th Percentile:         18
95th Percentile:         72
99th Percentile:         1196


In [28]:
print("\nSUBMISSIONS: NUM_COMMENTS DISTRIBUTION (Percentiles)")
print("=" * 80)

submissions_comments_percentiles = submissionsDF.select(
    F.expr("percentile_approx(num_comments, 0.25)").alias("p25"),
    F.expr("percentile_approx(num_comments, 0.50)").alias("p50_median"),
    F.expr("percentile_approx(num_comments, 0.75)").alias("p75"),
    F.expr("percentile_approx(num_comments, 0.90)").alias("p90"),
    F.expr("percentile_approx(num_comments, 0.95)").alias("p95"),
    F.expr("percentile_approx(num_comments, 0.99)").alias("p99")
).collect()[0]

print(f"25th Percentile:         {submissions_comments_percentiles['p25']}")
print(f"50th Percentile (Median): {submissions_comments_percentiles['p50_median']}")
print(f"75th Percentile:         {submissions_comments_percentiles['p75']}")
print(f"90th Percentile:         {submissions_comments_percentiles['p90']}")
print(f"95th Percentile:         {submissions_comments_percentiles['p95']}")
print(f"99th Percentile:         {submissions_comments_percentiles['p99']}")
print("=" * 80)


SUBMISSIONS: NUM_COMMENTS DISTRIBUTION (Percentiles)


[Stage 71:=====================>                                    (3 + 5) / 8]

25th Percentile:         0
50th Percentile (Median): 1
75th Percentile:         2
90th Percentile:         13
95th Percentile:         34
99th Percentile:         204


## Submissions Dataset Analysis

Submissions constitute a much smaller portion of the dataset, with approximately **2.0 million posts** compared to over **62 million comments**. As a result, submissions primarily function as organizational anchors that initiate discussion threads rather than as primary sites of interaction.

Given this structural role, submissions are not analyzed at the same level of granularity as comments. Instead, they are treated as contextual scaffolding for comment-level discourse, and their content is used only insofar as it provides temporal and topical context for downstream analyses.


### Missing/null values analysis for submissions

In [29]:
print("\nSUBMISSIONS: MISSING VALUES ANALYSIS")
print("=" * 80)

submissions_null_counts = submissionsDF.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in submissionsDF.columns
]).collect()[0]

for col_name in submissionsDF.columns:
    null_count = submissions_null_counts[col_name]
    null_pct = (null_count / submissions_count) * 100 if submissions_count > 0 else 0
    print(f"{col_name:30s} | Nulls: {null_count:12,} ({null_pct:6.2f}%)")
print("=" * 80)


SUBMISSIONS: MISSING VALUES ANALYSIS
id                             | Nulls:            0 (  0.00%)
author                         | Nulls:            0 (  0.00%)
created_utc                    | Nulls:            0 (  0.00%)
num_comments                   | Nulls:            0 (  0.00%)
score                          | Nulls:            0 (  0.00%)
selftext                       | Nulls:            0 (  0.00%)
title                          | Nulls:            0 (  0.00%)


### Top authors by submission count

In [30]:
print("\nSUBMISSIONS: TOP 20 AUTHORS BY SUBMISSION COUNT")
print("=" * 80)

top_submission_authors = submissionsDF.groupBy("author").agg(
    F.count("*").alias("submission_count"),
    F.avg("score").alias("avg_score"),
    F.avg("num_comments").alias("avg_num_comments")
).orderBy(F.desc("submission_count")).limit(20)

top_submission_authors.show(20, truncate=False)


SUBMISSIONS: TOP 20 AUTHORS BY SUBMISSION COUNT


[Stage 77:=====================>                                    (3 + 5) / 8]

+--------------------+----------------+------------------+--------------------+
|author              |submission_count|avg_score         |avg_num_comments    |
+--------------------+----------------+------------------+--------------------+
|[deleted]           |582970          |31.384767655282435|5.62501500934868    |
|Onboarding92        |1572            |1.0197201017811706|0.029262086513994912|
|WSBVoteBot          |1438            |1.4193324061196106|9.088317107093186   |
|OPINION_IS_UNPOPULAR|1361            |2149.7759000734754|10818.34680382072   |
|AutoModerator       |1018            |437.72593320235757|17977.240667976424  |
|predictany007       |409             |271.9046454767726 |66.53300733496333   |
|RealPennyMuncher    |407             |4.031941031941032 |4.356265356265356   |
|DaddyDersch         |379             |79.36147757255937 |44.99472295514512   |
|patrickbateman02    |346             |109.34971098265896|35.92774566473989   |
|ShopBitter          |305             |8

## Temporal Analysis

### Activity by year for the analysis period (2020-2022)

In [32]:
commentsDF_temporal = commentsDF.withColumn("created_ts", F.to_timestamp(F.col("created_utc")))
submissionsDF_temporal = submissionsDF.withColumn("created_ts", F.to_timestamp(F.col("created_utc")))

print("=" * 80)
print("COMMENTS: ACTIVITY BY YEAR")
print("=" * 80)
comments_by_year = commentsDF_temporal.withColumn("year", F.year("created_ts")).groupBy("year").count().orderBy("year")
comments_by_year.show(50)

print("\n" + "=" * 80)
print("SUBMISSIONS: ACTIVITY BY YEAR")
print("=" * 80)
submissions_by_year = submissionsDF_temporal.withColumn("year", F.year("created_ts")).groupBy("year").count().orderBy("year")
submissions_by_year.show(50)

COMMENTS: ACTIVITY BY YEAR


[Stage 80:=====================================================>  (51 + 2) / 53]

+----+--------+
|year|   count|
+----+--------+
|2019|    1515|
|2020|18202992|
|2021|30356739|
|2022|14229576|
+----+--------+


SUBMISSIONS: ACTIVITY BY YEAR
+----+-------+
|year|  count|
+----+-------+
|2019|     40|
|2020| 304008|
|2021|1407482|
|2022| 244310|
+----+-------+



### Activity by month for the analysis period (2020-2022)

In [33]:
print("\n" + "=" * 80)
print("COMMENTS: ACTIVITY BY YEAR-MONTH (2020-2022)")
print("=" * 80)

t0 = F.to_timestamp(F.lit("2020-01-01"))
t1 = F.to_timestamp(F.lit("2022-12-31 23:59:59"))

comments_by_month = (commentsDF_temporal
    .filter((F.col("created_ts") >= t0) & (F.col("created_ts") <= t1))
    .withColumn("year", F.year("created_ts"))
    .withColumn("month", F.month("created_ts"))
    .groupBy("year", "month")
    .count()
    .orderBy("year", "month"))

comments_by_month.show(50)

print("\n" + "=" * 80)
print("SUBMISSIONS: ACTIVITY BY YEAR-MONTH (2020-2022)")
print("=" * 80)

submissions_by_month = (submissionsDF_temporal
    .filter((F.col("created_ts") >= t0) & (F.col("created_ts") <= t1))
    .withColumn("year", F.year("created_ts"))
    .withColumn("month", F.month("created_ts"))
    .groupBy("year", "month")
    .count()
    .orderBy("year", "month"))

submissions_by_month.show(50)


COMMENTS: ACTIVITY BY YEAR-MONTH (2020-2022)


+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2020|    1| 516344|
|2020|    2|1048720|
|2020|    3|2608503|
|2020|    4|1917114|
|2020|    5|1547086|
|2020|    6|1816607|
|2020|    7|1507998|
|2020|    8|1356031|
|2020|    9|1403769|
|2020|   10|1195029|
|2020|   11|1539777|
|2020|   12|1746014|
|2021|    1|8255151|
|2021|    2|6595431|
|2021|    3|4181712|
|2021|    4|1677195|
|2021|    5|1365959|
|2021|    6|2186741|
|2021|    7|1097351|
|2021|    8|1013862|
|2021|    9|1006741|
|2021|   10| 919230|
|2021|   11|1057234|
|2021|   12|1000132|
|2022|    1|1373589|
|2022|    2|1157150|
|2022|    3|1127085|
|2022|    4|1094739|
|2022|    5|1262054|
|2022|    6|1133208|
|2022|    7|1083179|
|2022|    8|1723484|
|2022|    9|1196243|
|2022|   10|1162684|
|2022|   11|1004152|
|2022|   12| 912009|
+----+-----+-------+


SUBMISSIONS: ACTIVITY BY YEAR-MONTH (2020-2022)
+----+-----+------+
|year|month| count|
+----+-----+------+
|2020|    1| 11249|
|2020|    2| 28501|
|2020|    

## Data Quality Checks

### Deleted/Removed Content Analysis in Comments

In [34]:
print("=" * 80)
print("COMMENTS: DELETED/REMOVED CONTENT")
print("=" * 80)

deleted_authors_count = commentsDF.filter(
    F.lower(F.col("author")).isin("[deleted]", "[removed]")
).count()

deleted_body_count = commentsDF.filter(
    F.lower(F.col("body")).isin("[deleted]", "[removed]")
).count()

print(f"Comments with [deleted]/[removed] author:  {deleted_authors_count:,} ({deleted_authors_count/comments_count*100:.2f}%)")
print(f"Comments with [deleted]/[removed] body:    {deleted_body_count:,} ({deleted_body_count/comments_count*100:.2f}%)")
print("=" * 80)

COMMENTS: DELETED/REMOVED CONTENT


[Stage 95:===================================================>    (49 + 4) / 53]

Comments with [deleted]/[removed] author:  12,135,451 (19.33%)
Comments with [deleted]/[removed] body:    10,944,265 (17.43%)


### Deleted/Removed Content Analysis in Submissions

In [35]:
print("\n" + "=" * 80)
print("SUBMISSIONS: DELETED/REMOVED CONTENT")
print("=" * 80)

deleted_sub_authors_count = submissionsDF.filter(
    F.lower(F.col("author")).isin("[deleted]", "[removed]")
).count()

deleted_selftext_count = submissionsDF.filter(
    F.lower(F.col("selftext")).isin("[deleted]", "[removed]")
).count()

print(f"Submissions with [deleted]/[removed] author:   {deleted_sub_authors_count:,} ({deleted_sub_authors_count/submissions_count*100:.2f}%)")
print(f"Submissions with [deleted]/[removed] selftext: {deleted_selftext_count:,} ({deleted_selftext_count/submissions_count*100:.2f}%)")
print("=" * 80)


SUBMISSIONS: DELETED/REMOVED CONTENT
Submissions with [deleted]/[removed] author:   582,970 (29.81%)
Submissions with [deleted]/[removed] selftext: 1,292,471 (66.08%)


### Bot Account Detection in Comments

In [38]:
print("=" * 80)
print("COMMENTS: (Possible) BOT ACCOUNT DETECTION")
print("=" * 80)

bot_comments_count = commentsDF.filter(
    F.coalesce(
        F.lower(F.col("author")).contains("automoderator") |
        F.lower(F.col("author")).contains("moderator") |
        F.lower(F.col("author")).contains("bot") |
        F.lower(F.col("author")).contains("automod"),
        F.lit(False)
    )
).count()

print(f"Comments from bot accounts: {bot_comments_count:,} ({bot_comments_count/comments_count*100:.2f}%)")
print("=" * 80)

print("\nTOP (possible) BOT ACCOUNTS IN COMMENTS:")
commentsDF.filter(
    F.coalesce(
        F.lower(F.col("author")).contains("automoderator") |
        F.lower(F.col("author")).contains("moderator") |
        F.lower(F.col("author")).contains("bot") |
        F.lower(F.col("author")).contains("automod"),
        F.lit(False)
    )
).groupBy("author").count().orderBy(F.desc("count")).show(20, truncate=False)

COMMENTS: (Possible) BOT ACCOUNT DETECTION


Comments from bot accounts: 780,926 (1.24%)

TOP (possible) BOT ACCOUNTS IN COMMENTS:


[Stage 119:=================================================>     (48 + 5) / 53]

+------------------+------+
|author            |count |
+------------------+------+
|AutoModerator     |526362|
|WSBVoteBot        |26632 |
|RemindMeBot       |15285 |
|RobotArtichoke    |9736  |
|tu_test_bot       |6241  |
|neutralpoliticsbot|6212  |
|halfmanhalfrobot69|4384  |
|irobot001         |4199  |
|FullCuntalLobotomy|4131  |
|SuperBottit       |4112  |
|pickbot           |3765  |
|RichieBot         |3473  |
|Both-Employee-3421|3196  |
|TROLOLOLBOT       |3154  |
|Ok_Bottle_2198    |2810  |
|OofStickyBot      |2678  |
|Phlebotanist      |2538  |
|remindditbot      |2107  |
|NaNoBoT900        |2086  |
|RealisticHumanBot |2048  |
+------------------+------+
only showing top 20 rows



### Bot Account Detection in Submissions

In [39]:
print("\n" + "=" * 80)
print("SUBMISSIONS: (Possible) BOT ACCOUNT DETECTION")
print("=" * 80)

bot_submissions_count = submissionsDF.filter(
    F.coalesce(
        F.lower(F.col("author")).contains("automoderator") |
        F.lower(F.col("author")).contains("moderator") |
        F.lower(F.col("author")).contains("bot") |
        F.lower(F.col("author")).contains("automod"),
        F.lit(False)
    )
).count()

print(f"Submissions from bot accounts: {bot_submissions_count:,} ({bot_submissions_count/submissions_count*100:.2f}%)")
print("=" * 80)

print("\nTOP (Possible) BOT ACCOUNTS IN SUBMISSIONS:")
submissionsDF.filter(
    F.coalesce(
        F.lower(F.col("author")).contains("automoderator") |
        F.lower(F.col("author")).contains("moderator") |
        F.lower(F.col("author")).contains("bot") |
        F.lower(F.col("author")).contains("automod"),
        F.lit(False)
    )
).groupBy("author").count().orderBy(F.desc("count")).show(20, truncate=False)


SUBMISSIONS: (Possible) BOT ACCOUNT DETECTION
Submissions from bot accounts: 5,294 (0.27%)

TOP (Possible) BOT ACCOUNTS IN SUBMISSIONS:
+--------------------+-----+
|author              |count|
+--------------------+-----+
|WSBVoteBot          |1438 |
|AutoModerator       |1018 |
|TheModeratorWrangler|72   |
|Bottom_Wobbles      |32   |
|No-Bottle8391       |31   |
|orbotron88          |26   |
|Bebotron88          |24   |
|robot_handjob       |20   |
|RetardStockBot      |18   |
|DisabledSexRobot    |16   |
|John_Bot            |16   |
|Jess_Bot            |15   |
|robot-legs          |14   |
|pickbot             |14   |
|autobot12349876     |13   |
|cazbot              |13   |
|piss-in-a-bottle    |13   |
|iaintabotdotcom     |12   |
|Deepbotz            |12   |
|JimboTheSimpleton   |11   |
+--------------------+-----+
only showing top 20 rows



## Summary of Exploratory Findings

Overall, the exploratory analysis reveals a communication environment characterized by **extreme scale**, **severe engagement imbalance**, and **short-form, reactive discourse**. Attention is unevenly distributed, with rare but intense episodes of collective amplification shaping aggregate patterns.

These properties motivate analytical choices in subsequent sections, including the use of clustering to identify discourse regimes, sentiment aggregation rather than document-level inference, and robust statistical summaries that account for heavy-tailed behavior.